# Phase 2 — Settlement-Scale Diagnostics (GHSL-SMOD)

This notebook evaluates VIIRS DNB-BRDF radiance dynamics across GHSL-SMOD settlement classes in Samar–Leyte.

The objective is to:

- Compare radiance behavior across settlement hierarchy (Water → Rural → Urban Centre)
- Assess class-level observability using valid pixel thresholds
- Examine disaster signal response (e.g., Typhoon Yolanda, 2013)
- Filter extreme outliers (0–95 percentile) for robust visualization

All analyses are restricted to high-quality observations (Valid % > threshold).

In [7]:
import pandas as pd
import plotly.express as px

import numpy as np
import plotly.graph_objects as go


In [48]:
# -----------------------------------------------------
# Load Data
# -----------------------------------------------------

data_path = "/Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/Research/powerwatch-ph/Datasets/SamarLeyte_DNBBRDF_AllClasses_DailyStats_2012_2024.csv"

df = pd.read_csv(data_path)
df["date"] = pd.to_datetime(df["date"])

# Restrict to 2013
df_2013 = df[df["date"].dt.year == 2013].copy()
df_2013.sort_values("date", inplace=True)

df_2013.head()

,system:index,ClassCode,Mean_by_sum,NTL_max,NTL_mean,NTL_median,NTL_min,NTL_p05,NTL_p25,NTL_p75,NTL_p95,NTL_std,NTL_sum,Valid_px,date,.geo
263,0_2013_01_01,10,0.137101,9.376099,0.237317,0.005841,0.0,0.005841,0.005841,0.226290,1.340240,0.635664,85.413749,623,2013-01-01,"{""type"":""MultiPoint"",""coordinates"":[]}"
8050,2_2013_01_01,12,0.290810,5.525498,0.296907,0.005055,0.0,0.005055,0.005055,0.483381,1.297424,0.458463,1836.466972,6315,2013-01-01,"{""type"":""MultiPoint"",""coordinates"":[]}"
11979,3_2013_01_01,13,0.257046,4.996292,0.280857,0.004575,0.0,0.004575,0.004575,0.452367,1.238996,0.453080,397.649987,1547,2013-01-01,"{""type"":""MultiPoint"",""coordinates"":[]}"
27162,7_2013_01_01,30,4.011389,41.251404,4.304605,3.372896,0.0,0.075490,0.621310,6.114589,13.473122,4.914043,1432.065975,357,2013-01-01,"{""type"":""MultiPoint"",""coordinates"":[]}"
19718,5_2013_01_01,22,0.268929,4.576447,0.304333,0.003987,0.0,0.003987,0.003987,0.451575,1.292767,0.556588,178.299996,663,2013-01-01,"{""type"":""MultiPoint"",""coordinates"":[]}"


In [5]:
df['date'] = pd.to_datetime(df['date'])
df_2013 = df[df['date'].dt.year == 2013]
df_2013

,system:index,ClassCode,Mean_by_sum,NTL_max,NTL_mean,NTL_median,NTL_min,NTL_p05,NTL_p25,NTL_p75,NTL_p95,NTL_std,NTL_sum,Valid_px,date,.geo
263,0_2013_01_01,10,0.137101,9.376099,0.237317,0.005841,0.000000,0.005841,0.005841,0.226290,1.340240,0.635664,85.413749,623,2013-01-01,"{""type"":""MultiPoint"",""coordinates"":[]}"
264,0_2013_01_04,10,0.202113,1.163504,0.503487,0.257449,0.003080,0.135847,0.159172,0.863650,1.163504,0.354094,4.446477,22,2013-01-04,"{""type"":""MultiPoint"",""coordinates"":[]}"
265,0_2013_01_05,10,0.064178,1.072734,0.156582,0.122727,0.001920,0.010611,0.042256,0.217118,0.417952,0.153724,13.284882,207,2013-01-05,"{""type"":""MultiPoint"",""coordinates"":[]}"
266,0_2013_01_09,10,0.029872,0.224832,0.101046,0.098338,0.008514,0.047102,0.061866,0.132283,0.224832,0.040952,1.314389,44,2013-01-09,"{""type"":""MultiPoint"",""coordinates"":[]}"
267,0_2013_01_11,10,0.068537,0.743498,0.159201,0.169581,0.010000,0.024574,0.108151,0.206790,0.271686,0.111928,3.426871,50,2013-01-11,"{""type"":""MultiPoint"",""coordinates"":[]}"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27381,7_2013_12_25,30,1.810858,29.673061,1.972888,0.941304,0.001508,0.190278,0.439685,1.799443,8.877767,3.238420,664.584817,367,2013-12-25,"{""type"":""MultiPoint"",""coordinates"":[]}"
27382,7_2013_12_26,30,1.718405,23.436907,1.876174,0.942375,0.001304,0.052926,0.570127,1.808637,7.439327,2.935782,740.632650,431,2013-12-26,"{""type"":""MultiPoint"",""coordinates"":[]}"
27383,7_2013_12_27,30,0.566044,1.749305,0.630235,0.589643,0.197282,0.292740,0.408475,0.764658,1.214776,0.288327,122.265509,216,2013-12-27,"{""type"":""MultiPoint"",""coordinates"":[]}"
27384,7_2013_12_28,30,0.332300,1.316310,0.368232,0.373976,0.010000,0.136744,0.236698,0.448921,0.636137,0.181697,65.130873,196,2013-12-28,"{""type"":""MultiPoint"",""coordinates"":[]}"


In [6]:
# -----------------------------------------------------
# GHSL-SMOD Class Metadata
# -----------------------------------------------------

GHSL_INFO = {
    "10": {"label": "Water (10)", "color": "#7ab6f5"},
    "11": {"label": "Very Low Density Rural (11)", "color": "#cdf57a"},
    "12": {"label": "Low Density Rural (12)", "color": "#abcd66"},
    "13": {"label": "Rural Cluster (13)", "color": "#375623"},
    "21": {"label": "Suburban / Peri-urban (21)", "color": "#ffff00"},
    "22": {"label": "Semi-dense Urban Cluster (22)", "color": "#a87000"},
    "23": {"label": "Dense Urban Cluster (23)", "color": "#732600"},
    "30": {"label": "Urban Centre (30)", "color": "#ff0000"},
}

In [43]:
def plot_timeseries_by_class_points(df, value_col="Mean_by_sum", threshold=10):

    fig = go.Figure()
    df_plot = df.copy()
    df_plot["ClassCode"] = df_plot["ClassCode"].astype(str)

    for cls in sorted(df_plot["ClassCode"].unique(), key=lambda x: int(x)):

        df_cls = df_plot[df_plot["ClassCode"] == cls].copy()
        if df_cls.empty:
            continue

        # Percentile filter (0–95%)
        if df_cls[value_col].notna().sum() > 0:
            ntl_0 = np.nanpercentile(df_cls[value_col], 0)
            ntl_95 = np.nanpercentile(df_cls[value_col], 95)
            df_cls = df_cls[
                (df_cls[value_col] >= ntl_0) &
                (df_cls[value_col] <= ntl_95)
            ]

        # Valid threshold
        total_px = df_cls["Valid_px"].max()
        if total_px and total_px > 0:
            df_cls["Valid_pct"] = df_cls["Valid_px"] / total_px * 100
            df_cls = df_cls[df_cls["Valid_pct"] > threshold]

        if df_cls.empty:
            continue

        fig.add_trace(
            go.Scatter(
                x=df_cls["date"],
                y=df_cls[value_col],
                mode="markers",
                marker=dict(
                    color=GHSL_INFO.get(cls, {"color": "#999999"})["color"],
                    size=6,
                    opacity=0.6
                ),
                name=GHSL_INFO.get(cls, {"label": f"Class {cls}"})["label"]
            )
        )

    # -------------------------
    # Haiyan marker (consistent)
    # -------------------------
    haiyan_date = pd.to_datetime("2013-11-08")

    fig.add_vline(
        x=haiyan_date,
        line_dash="dash",
        line_color="black",
        line_width=3
    )

    fig.add_annotation(
        x=haiyan_date,
        y=1.05,
        yref="paper",
        text="Haiyan",
        showarrow=False,
        font=dict(size=18, color="black"),
        bgcolor="white",
        bordercolor="black",
        borderwidth=1,
        xanchor="left"
    )

    # -------------------------
    # Layout (PROPER SCALE)
    # -------------------------
    fig.update_layout(
        xaxis=dict(
            title="Date",
            title_font=dict(size=22),
            tickfont=dict(size=18)
        ),
        yaxis=dict(
            title="Radiance (nW·cm⁻²·sr⁻¹)",
            title_font=dict(size=22),
            tickfont=dict(size=18)
        ),
        legend=dict(
            orientation="h",
            y=1.2,
            x=0.02,
            xanchor="left",
            font=dict(size=16)
        ),
        template="plotly_white",
        height=500,
        width=1300,
        paper_bgcolor='rgba(0,0,0,0)'
    )

    fig.show()

# Run
plot_timeseries_by_class_points(df_2013, value_col="Mean_by_sum", threshold=10)

## Daily Scatter by GHSL Class (Raw Points)

This figure shows daily DNB-BRDF radiance values for 2013 grouped by GHSL settlement class.  
Each dot represents one daily class-level mean radiance value after percentile (0–95%) and valid-pixel filtering.

**Key observations:**

- Urban Centre (30) exhibits the highest radiance and the largest variability.
- Dense Urban (23) and Semi-dense Urban (22) form a clear intermediate tier.
- Rural classes (11–13) remain low and tightly clustered.
- After Haiyan (8 Nov 2013), urban radiance collapses sharply, while rural and water classes show minimal structural change.

This plot emphasizes dispersion and class separation without smoothing.

In [42]:
# --- Unified GHSL Info ---
GHSL_INFO = {
    "10": {"label": "Water (10)", "color": "#7ab6f5"},
    "11": {"label": "Very Low Density Rural (11)", "color": "#cdf57a"},
    "12": {"label": "Low Density Rural (12)", "color": "#abcd66"},
    "13": {"label": "Rural Cluster (13)", "color": "#375623"},
    "21": {"label": "Suburban / Peri-urban (21)", "color": "#ffff00"},
    "22": {"label": "Semi-dense Urban Cluster (22)", "color": "#a87000"},
    "23": {"label": "Dense Urban Cluster (23)", "color": "#732600"},
    "30": {"label": "Urban Centre (30)", "color": "#ff0000"},
}


def plot_timeseries_by_class(
    df,
    value_col="Mean_by_sum",
    w=4,              # ← default 4-day rolling
    threshold=10
):

    fig = go.Figure()
    df_plot = df.copy()
    df_plot["date"] = pd.to_datetime(df_plot["date"])
    df_plot["ClassCode"] = df_plot["ClassCode"].astype(str)
    df_plot = df_plot.sort_values("date")

    for cls in sorted(df_plot["ClassCode"].unique(), key=lambda x: int(x)):

        df_cls = df_plot[df_plot["ClassCode"] == cls].copy()
        if df_cls.empty:
            continue

        # 0–95 percentile filter
        if df_cls[value_col].notna().sum() > 0:
            p0 = np.nanpercentile(df_cls[value_col], 0)
            p95 = np.nanpercentile(df_cls[value_col], 95)
            df_cls = df_cls[
                (df_cls[value_col] >= p0) &
                (df_cls[value_col] <= p95)
            ]

        # Valid pixel % filter
        total_px = df_cls["Valid_px"].max()
        if total_px and total_px > 0:
            df_cls["Valid_pct"] = df_cls["Valid_px"] / total_px * 100
            df_cls = df_cls[df_cls["Valid_pct"] > threshold]

        if df_cls.empty:
            continue

        # 4-day rolling mean
        df_cls[f"MA_{w}d"] = df_cls[value_col].rolling(w, min_periods=1).mean()

        color = GHSL_INFO.get(cls, {"color": "#999999"})["color"]
        label = GHSL_INFO.get(cls, {"label": f"Class {cls}"})["label"]

        # --- Raw points (faded)
        fig.add_trace(
            go.Scatter(
                x=df_cls["date"],
                y=df_cls[value_col],
                mode="markers",
                marker=dict(size=8, color=color, opacity=0.8),
                showlegend=False
            )
        )

        # --- Rolling mean line
        fig.add_trace(
            go.Scatter(
                x=df_cls["date"],
                y=df_cls[f"MA_{w}d"],
                mode="lines",
                line=dict(color=color, width=2),
                name=label
            )
        )

    # -------------------------
    # Haiyan (consistent style)
    # -------------------------
    haiyan_date = pd.to_datetime("2013-11-08")

    fig.add_vline(
        x=haiyan_date,
        line_dash="dash",
        line_color="black",
        line_width=3
    )

    fig.add_annotation(
        x=haiyan_date,
        y=1.05,
        yref="paper",
        text="Haiyan",
        showarrow=False,
        font=dict(size=18, color="black"),
        bgcolor="white",
        bordercolor="black",
        borderwidth=1,
        xanchor="left"
    )

    # -------------------------
    # Layout (uniform thesis scale)
    # -------------------------
    fig.update_layout(
        xaxis=dict(
            title="Date",
            title_font=dict(size=22),
            tickfont=dict(size=18)
        ),
        yaxis=dict(
            title="Radiance (nW·cm⁻²·sr⁻¹)",
            title_font=dict(size=22),
            tickfont=dict(size=18)
        ),
        legend=dict(
            orientation="h",
            y=1.2,
            x=0.05,
            xanchor="left",
            font=dict(size=16)
        ),
        template="plotly_white",
        height=500,
        width=1300,
        paper_bgcolor='rgba(0,0,0,0)'
    )

    fig.show()


# Run
plot_timeseries_by_class(df_2013, value_col="Mean_by_sum", w=1, threshold=40)

## Class-wise Time Series

This figure shows class-level radiance trajectories with minimal smoothing (w = 1), preserving daily variability.

**Key observations:**

- Clear radiance stratification by settlement intensity.
- Urban Centre (30) shows strong pre-disaster variability followed by a clear structural break at Haiyan.
- Dense and semi-dense classes exhibit moderate but distinct drops.
- Rural and water classes remain comparatively stable.

The vertical dashed line marks Haiyan landfall, highlighting a synchronized urban-scale radiance collapse.

## Settlement-Scale Reliability Diagnostics (GHSL-Based)

This cell computes settlement-scale observability and variability metrics for each GHSL-SMOD class (2012–2024), using daily VIIRS DNB-BRDF statistics.

### Data Scope
- Study period: **2013** (date range can be modified)
- Unit of analysis: GHSL settlement class (SMOD level)
- Metrics computed per class and per umbrella group

---

### Metrics Computed

**1. Spatial Completeness (%)**  
Average proportion of valid pixels relative to total class pixels across the full time range.

**2. Temporal Completeness (%)**  
Fraction of days meeting pixel coverage thresholds:
- ≥25% valid pixels
- ≥50% valid pixels
- ≥75% valid pixels

**3. Maximum Dropout (days)**  
Longest consecutive run of days with <50% valid pixels.

**4. Temporal CV**  
Coefficient of variation of class-level daily mean radiance (signal volatility).

**5. Spatial CV**  
Median of per-day (NTL_std / NTL_mean), representing intra-class spatial heterogeneity.

---

### Group Structure

Classes are aggregated into umbrella groups:

- **Urban Centre (30)**
- **Urban Cluster (21, 22, 23)**
- **Rural Grid Cells (11, 12, 13)**
- **Water Bodies (10)**

Group averages are reported alongside individual class metrics.

This table forms the basis for settlement-scale reliability gating and cross-class comparability analysis.

In [53]:
DATE_RANGE = pd.date_range("2013-01-01", "2014-01-01", freq="D")

# -----------------------------
# 2) Pixel Counts
# -----------------------------
pixel_dict = {
    "10": {"count": 834.0,  "pct": 0.99},
    "11": {"count": 53864.0, "pct": 63.78},
    "12": {"count": 13773.0, "pct": 16.39},
    "13": {"count": 3364.0,  "pct": 3.98},
    "21": {"count": 9406.0,  "pct": 11.14},
    "22": {"count": 1322.0,  "pct": 1.57},
    "23": {"count": 1418.0,  "pct": 1.68},
    "30": {"count": 467.0,   "pct": 0.55},
}

# -----------------------------
# 3) Umbrella Groups
# -----------------------------
umbrella_map = {
    "Urban Centre": ["30"],
    "Urban Cluster": ["21", "22", "23"],
    "Rural Grid Cells": ["13", "12", "11"],
    "Water Bodies": ["10"],
}

class_to_group = {
    cls: group for group, classes in umbrella_map.items() for cls in classes
}

group_order = ["Urban Centre", "Urban Cluster", "Rural Grid Cells", "Water Bodies"]

class_order_within = {
    "Urban Centre": ["30"],
    "Urban Cluster": ["21", "22", "23"],
    "Rural Grid Cells": ["13", "12", "11"],
    "Water Bodies": ["10"]
}

# -----------------------------
# 4) Per-Class Metrics
# -----------------------------
rows = []

for cls, sub in df.groupby("ClassCode"):

    px_info = pixel_dict.get(cls, {"count": np.nan, "pct": np.nan})
    pixel_count = px_info["count"]

    s = (
        sub.set_index("date")
           .reindex(DATE_RANGE)
           .rename_axis("date")
           .reset_index()
    )

    s[["Valid_px", "NTL_mean", "NTL_std"]] = (
        s[["Valid_px", "NTL_mean", "NTL_std"]].fillna(0)
    )

    # Spatial completeness
    spatial = (s["Valid_px"] / pixel_count).mean() * 100 if pixel_count > 0 else 0

    # Temporal completeness
    temporal_25 = (s["Valid_px"] >= pixel_count * 0.25).mean() * 100
    temporal_50 = (s["Valid_px"] >= pixel_count * 0.50).mean() * 100
    temporal_75 = (s["Valid_px"] >= pixel_count * 0.75).mean() * 100

    # Max dropout (≥50% threshold)
    z = (s["Valid_px"] >= pixel_count * 0.50).astype(int)
    grp = (z != z.shift()).cumsum()
    dropout_lengths = z[z.eq(0)].groupby(grp[z.eq(0)]).size()
    max_dropout = int(dropout_lengths.max()) if not dropout_lengths.empty else 0

    # Variability
    v = s[(s["Valid_px"] > 0) & (s["NTL_mean"] > 0)]

    temporal_cv = (
        v["NTL_mean"].std() / v["NTL_mean"].mean()
        if not v.empty else np.nan
    )

    spatial_cv = (
        (v["NTL_std"] / v["NTL_mean"]).median()
        if not v.empty else np.nan
    )

    rows.append({
        "Group": class_to_group.get(cls),
        "ClassCode": cls,
        "Pixel_count": pixel_count,
        "Pixel_share(%)": px_info["pct"],
        "Spatial_completeness(%)": spatial,
        "Temporal_completeness ≥25%(%)": temporal_25,
        "Temporal_completeness ≥50%(%)": temporal_50,
        "Temporal_completeness ≥75%(%)": temporal_75,
        "Max_dropout_days": max_dropout,
        "Temporal_CV": temporal_cv,
        "Spatial_CV": spatial_cv,
    })

class_stats = pd.DataFrame(rows)

# -----------------------------
# 5) Group-Level Averages
# -----------------------------
avg_rows = []

for g in group_order:
    sub = class_stats[class_stats["Group"] == g]
    if sub.empty:
        continue

    avg_rows.append({
        "Group": f"{g} (average)",
        "ClassCode": "avg",
        "Pixel_count": sub["Pixel_count"].sum(),
        "Pixel_share(%)": sub["Pixel_share(%)"].sum(),
        "Spatial_completeness(%)": sub["Spatial_completeness(%)"].mean(),
        "Temporal_completeness ≥25%(%)": sub["Temporal_completeness ≥25%(%)"].mean(),
        "Temporal_completeness ≥50%(%)": sub["Temporal_completeness ≥50%(%)"].mean(),
        "Temporal_completeness ≥75%(%)": sub["Temporal_completeness ≥75%(%)"].mean(),
        "Max_dropout_days": sub["Max_dropout_days"].max(),
        "Temporal_CV": sub["Temporal_CV"].mean(),
        "Spatial_CV": sub["Spatial_CV"].mean(),
    })

group_stats = pd.DataFrame(avg_rows)

# -----------------------------
# 6) Final Assembly + Ordering
# -----------------------------
summary = pd.concat([group_stats, class_stats], ignore_index=True)

summary["order_key"] = summary.apply(
    lambda r: (
        group_order.index(r["Group"].replace(" (average)", "")),
        0 if r["ClassCode"] == "avg" else 1,
        class_order_within.get(
            r["Group"].replace(" (average)", r["Group"]), []
        ).index(r["ClassCode"])
        if r["ClassCode"] in class_order_within.get(
            r["Group"].replace(" (average)", r["Group"]), []
        ) else 99
    ),
    axis=1
)

summary = (
    summary.sort_values("order_key")
           .drop(columns="order_key")
           .reset_index(drop=True)
)

summary = summary.round({
    "Pixel_count": 0,
    "Pixel_share(%)": 2,
    "Spatial_completeness(%)": 1,
    "Temporal_completeness ≥25%(%)": 1,
    "Temporal_completeness ≥50%(%)": 1,
    "Temporal_completeness ≥75%(%)": 1,
    "Temporal_CV": 2,
    "Spatial_CV": 2
})

summary

,Group,ClassCode,Pixel_count,Pixel_share(%),Spatial_completeness(%),Temporal_completeness ≥25%(%),Temporal_completeness ≥50%(%),Temporal_completeness ≥75%(%),Max_dropout_days,Temporal_CV,Spatial_CV
0,Urban Centre (average),avg,467.0,0.55,31.9,43.2,32.0,20.8,22,0.71,1.39
1,Urban Centre,30,467.0,0.55,31.9,43.2,32.0,20.8,22,0.71,1.39
2,Urban Cluster (average),avg,12146.0,14.39,34.4,45.8,34.0,20.2,23,0.66,1.43
3,Urban Cluster,21,9406.0,11.14,33.7,45.6,34.4,20.5,22,0.65,1.52
4,Urban Cluster,22,1322.0,1.57,34.7,46.2,34.4,20.2,23,0.63,1.02
5,Urban Cluster,23,1418.0,1.68,34.8,45.6,33.1,19.9,23,0.69,1.76
6,Rural Grid Cells (average),avg,71001.0,84.15,34.2,48.5,35.2,19.4,23,0.68,1.24
7,Rural Grid Cells,13,3364.0,3.98,35.2,50.0,35.8,20.8,23,0.76,1.52
8,Rural Grid Cells,12,13773.0,16.39,33.5,46.7,35.0,19.4,23,0.65,1.33
9,Rural Grid Cells,11,53864.0,63.78,33.9,48.9,35.0,18.0,23,0.63,0.88


## Settlement-Scale Reliability Summary (2013)

This table summarizes spatial completeness, temporal completeness, dropout duration, and variability metrics for each GHSL settlement class and umbrella group across the full study period.

### Spatial Completeness (%)

- Urban Centre: ~31.9%  
- Urban Cluster classes: ~33–35%  
- Rural Grid Cells: ~33–35%  
- Water: ~58.5%

Water exhibits the highest spatial completeness. Land-based classes show broadly similar spatial coverage (~34%), indicating cloud masking affects urban and rural areas comparably.

---

### Temporal Completeness (≥50% Threshold)

- Urban Centre: ~32%  
- Urban Cluster (avg): ~34%  
- Rural Grid Cells (avg): ~35%  
- Water: ~43%

Rural classes slightly outperform dense urban in meeting ≥50% daily coverage. Urban Centre shows marginally lower temporal reliability due to smaller pixel pools and greater sensitivity to cloud loss.

---

### Maximum Dropout Duration

- Land classes: 22–23 consecutive days  
- Water: ~20 days  

All land-based classes experience prolonged multi-week observation gaps, consistent with persistent tropical cloud regimes.



## Annual Variability Diagnostics (Settlement-Scale CV)

This cell evaluates inter-annual radiance variability for each GHSL settlement class (2012–2024).

### Metrics

**Annual CV (Coefficient of Variation)**  
$$
CV = \frac{\sigma_{\text{annual}}}{\mu_{\text{annual}}}
$$

where
	•	$\mu_{\text{annual}}$ = annual mean of daily NTL
	•	$\sigma_{\text{annual}}$ = annual standard deviation


Computed using daily NTL_mean values per year and per class.

---

### Outputs

**1. Annual Line Plot**
- Shows year-to-year volatility trends.
- Log-scale highlights cross-class variability differences.
- Urban classes typically exhibit higher radiometric volatility.

**2. CV Distribution (Box Plot)**
- Summarizes long-term variability structure.
- Median CV annotated per class.
- Enables settlement-scale reliability comparison.

These diagnostics quantify structural signal stability across the urban–rural gradient.

In [55]:
# -----------------------------
# Annual CV Calculation
# -----------------------------
annual_cv = (
    df[df["NTL_mean"] > 0]
    .assign(year=df["date"].dt.year)
    .groupby(["ClassCode", "year"])
    .agg(
        mean_ntl=("NTL_mean", "mean"),
        std_ntl=("NTL_mean", "std")
    )
    .reset_index()
)

annual_cv["CV"] = annual_cv["std_ntl"] / annual_cv["mean_ntl"]

# -----------------------------
# Use Existing GHSL_INFO
# -----------------------------
annual_cv["ClassLabel"] = annual_cv["ClassCode"].map(
    lambda x: GHSL_INFO.get(x, {}).get("label", f"Class {x}")
)

color_map = {
    GHSL_INFO[k]["label"]: GHSL_INFO[k]["color"]
    for k in GHSL_INFO
}

ordered_labels = [
    GHSL_INFO[k]["label"]
    for k in sorted(GHSL_INFO.keys(), key=int)
]

# -----------------------------
# Line Plot
# -----------------------------
fig_line = px.line(
    annual_cv,
    x="year",
    y="CV",
    color="ClassLabel",
    color_discrete_map=color_map,
    category_orders={"ClassLabel": ordered_labels},
)

fig_line.update_traces(mode="lines+markers", line=dict(width=2))

fig_line.update_layout(
    template="plotly_white",
    width=1000,
    height=500,
    legend=dict(orientation="h", y=1.15, x=0.5, xanchor="center", font=dict(size=16)),
    xaxis=dict(title="Year", title_font=dict(size=22), tickfont=dict(size=18)),
    yaxis=dict(title="NTL Coefficient of Variation", type="log",
               title_font=dict(size=22), tickfont=dict(size=18)),
)

fig_line.show()

# -----------------------------
# Box Plot
# -----------------------------
fig_box = px.box(
    annual_cv,
    x="ClassLabel",
    y="CV",
    color="ClassLabel",
    color_discrete_map=color_map,
    category_orders={"ClassLabel": ordered_labels},
    points="outliers"
)

fig_box.update_layout(
    template="plotly_white",
    width=1000,
    height=500,
    showlegend=False,
    xaxis=dict(title="GHSL Class", title_font=dict(size=22), tickfont=dict(size=16)),
    yaxis=dict(title="NTL Coefficient of Variation", type="log",
               title_font=dict(size=22), tickfont=dict(size=18)),
)

# Median annotations
medians = annual_cv.groupby("ClassLabel")["CV"].median()
for cls, val in medians.items():
    fig_box.add_annotation(
        x=cls,
        y=val,
        text=f"{val:.2f}",
        showarrow=False,
        font=dict(size=16),
        yshift=-35
    )

fig_box.show()

## Annual Variability Diagnostics (Settlement-Scale CV)

This cell evaluates inter-annual radiance variability for each GHSL settlement class (2012–2024).

### Metrics

**Annual CV (Coefficient of Variation)**  
$$
CV = \frac{\sigma_{\text{annual}}}{\mu_{\text{annual}}}
$$

where
	•	$\mu_{\text{annual}}$ = annual mean of daily NTL
	•	$\sigma_{\text{annual}}$ = annual standard deviation


Computed using daily NTL_mean values per year and per class.

---

### Outputs

**1. Annual Line Plot**
- Shows year-to-year volatility trends.
- Log-scale highlights cross-class variability differences.
- Urban classes typically exhibit higher radiometric volatility.

**2. CV Distribution (Box Plot)**
- Summarizes long-term variability structure.
- Median CV annotated per class.
- Enables settlement-scale reliability comparison.

These diagnostics quantify structural signal stability across the urban–rural gradient.

In [57]:
# -----------------------------
# 1) 30-day Moving Average
# -----------------------------
df = df.sort_values(["ClassCode", "date"]).copy()

df["NTL_MA30"] = (
    df.groupby("ClassCode")["NTL_mean"]
      .transform(lambda x: x.rolling(30, min_periods=1).mean())
)

# -----------------------------
# 2) Annual CV (MA30-based)
# -----------------------------
annual_cv = (
    df[df["NTL_MA30"] > 0]
    .assign(year=df["date"].dt.year)
    .groupby(["ClassCode", "year"])
    .agg(
        mean_ntl=("NTL_MA30", "mean"),
        std_ntl=("NTL_MA30", "std")
    )
    .reset_index()
)

annual_cv["CV"] = annual_cv["std_ntl"] / annual_cv["mean_ntl"]

# -----------------------------
# 3) Use Existing GHSL_INFO
# -----------------------------
annual_cv["ClassLabel"] = annual_cv["ClassCode"].map(
    lambda x: GHSL_INFO.get(x, {}).get("label", f"Class {x}")
)

color_map = {
    GHSL_INFO[k]["label"]: GHSL_INFO[k]["color"]
    for k in GHSL_INFO
}

ordered_labels = [
    GHSL_INFO[k]["label"]
    for k in sorted(GHSL_INFO.keys(), key=int)
]

# -----------------------------
# 4) Line Plot (Uniform Thesis Scale)
# -----------------------------
fig_line = px.line(
    annual_cv,
    x="year",
    y="CV",
    color="ClassLabel",
    color_discrete_map=color_map,
    category_orders={"ClassLabel": ordered_labels},
)

fig_line.update_traces(mode="lines+markers", line=dict(width=2))

fig_line.update_layout(
    template="plotly_white",
    width=1000,
    height=500,
    legend=dict(
        orientation="h",
        y=1.15,
        x=0.5,
        xanchor="center",
        font=dict(size=16)
    ),
    xaxis=dict(title="Year", title_font=dict(size=22), tickfont=dict(size=18)),
    yaxis=dict(title="NTL Coefficient of Variation (MA30)",
               type="log",
               title_font=dict(size=22),
               tickfont=dict(size=18)),
)

fig_line.show()


# -----------------------------
# 5) Box Plot
# -----------------------------
fig_box = px.box(
    annual_cv,
    x="ClassLabel",
    y="CV",
    color="ClassLabel",
    color_discrete_map=color_map,
    category_orders={"ClassLabel": ordered_labels},
    points="outliers"
)

fig_box.update_layout(
    template="plotly_white",
    width=1000,
    height=500,
    showlegend=False,
    xaxis=dict(title="GHSL Class", title_font=dict(size=22), tickfont=dict(size=16)),
    yaxis=dict(title="NTL Coefficient of Variation (MA30)",
               type="log",
               title_font=dict(size=22),
               tickfont=dict(size=18)),
)

# Median annotations
medians = annual_cv.groupby("ClassLabel")["CV"].median()

for cls, val in medians.items():
    fig_box.add_annotation(
        x=cls,
        y=val,
        text=f"{val:.2f}",
        showarrow=False,
        font=dict(size=16),
        yshift=-35
    )

fig_box.show()



### Variability Metrics

**Temporal CV (signal volatility)**  
- Urban Centre: ~0.71  
- Urban Cluster: ~0.66  
- Rural Grid Cells: ~0.68  
- Water: ~1.47  

Water shows the highest relative volatility due to low baseline radiance amplifying proportional variation.

**Spatial CV (intra-class heterogeneity)**  
- Urban Centre: ~1.39  
- Urban Cluster: ~1.02–1.76  
- Rural Grid Cells: ~0.88–1.52  
- Water: ~1.13  

Dense and peri-urban classes exhibit greater intra-class heterogeneity than rural classes.

---

### Interpretation

- Spatial completeness is structurally similar across land settlement types.
- Temporal reliability remains moderate under tropical cloud conditions.
- Water is highly observable but relatively volatile.
- Urban classes demonstrate stable but moderately heterogeneous radiometric behavior.

These diagnostics justify reliability qualification before settlement-scale inference.